# LC 269 — Alien Dictionary
**Difficulty:** Hard | **Pattern:** Topological Sort (Graph)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Adjacent words in sorted alien order reveal
character ordering. Find the first differing character between each
consecutive word pair — that gives you a directed edge in a precedence
graph. Topological sort (Kahn's BFS) on that graph yields the alien
alphabet order. A cycle means the input is contradictory → return "".
</div>

## Official Problem Statement

There is a new alien language that uses the English alphabet. However,
the order among the letters is unknown to you.

You are given a list of strings `words` from the alien language's
dictionary, where the strings in `words` are **sorted lexicographically**
by the rules of this new language.

Return *a string of the unique letters in the new alien language sorted
in lexicographically increasing order by the new language's rules.*
If there is no solution, return `""`. If there are multiple valid
solutions, return **any** of them.

**Constraints:**
- `1 <= words.length <= 100`
- `1 <= words[i].length <= 100`
- `words[i]` consists of only lowercase English letters.
- It is guaranteed that the input is from a valid alien language.

## What This Is Actually Asking

You are given words already sorted in an unknown order — like a
dictionary printed in that alien alphabet. Your job is to recover
the ordering of the alphabet itself.

Each adjacent word pair gives at most one ordering clue: the first
position where the two words differ tells you which letter comes
before which.

Once you have all the clues, the problem becomes: given a set of
"A comes before B" constraints, find a valid total ordering —
exactly what topological sort solves.

A contradiction (cycle) or prefix violation means the input is
invalid, so return `""`.

## Walk Through an Example by Hand

```
words = ["wrt", "wrf", "er", "ett", "rftt"]
```

**Step 1 — Compare adjacent pairs for first diff:**
```
"wrt" vs "wrf" → pos 2: t != f  →  t -> f
"wrf" vs "er"  → pos 0: w != e  →  w -> e
"er"  vs "ett" → pos 1: r != t  →  r -> t
"ett" vs "rftt"→ pos 0: e != r  →  e -> r
```

**Step 2 — Build graph + in-degrees:**
```
Edges: t->f, w->e, r->t, e->r
All chars: {w, r, t, f, e}
in_degree: w=0, r=0(+1=1 from e), t=0(+1=1 from r),
           f=1(from t), e=1(from w)
```

**Step 3 — Kahn's BFS (start with in_degree=0):**
```
Queue: [w, r?]  actually only w=0
Process w → e's in_degree drops to 0 → add e
Process e → r's in_degree drops to 0 → add r
Process r → t's in_degree drops to 0 → add t
Process t → f's in_degree drops to 0 → add f
Process f → done
Result: "w e r t f"
```

## The Picture

```
words (sorted in alien order)
  "wrt"
  "wrf"    ← diff at pos 2: t comes before f
  "er"     ← diff at pos 0: w comes before e
  "ett"    ← diff at pos 1: r comes before t
  "rftt"   ← diff at pos 0: e comes before r

Precedence Graph:

   w ──► e ──► r ──► t ──► f
                           ▲
   (only edges matter,     │
    not original chars)    │

Kahn's BFS Topological Sort:

  [ Queue ]        [ Visited ]      [ Output ]
  [  w    ]   →   pick w       →    "w"
  [  e    ]   →   pick e       →    "we"
  [  r    ]   →   pick r       →    "wer"
  [  t    ]   →   pick t       →    "wert"
  [  f    ]   →   pick f       →    "wertf"

  If len(output) != len(all_chars) → cycle → return ""

  Invalid prefix case:
  "abc", "ab"  ← longer word before shorter prefix → return ""
```

## When To Use This Pattern

- When you have items in a **sorted/ordered list** and need to infer
  the **ordering rules** from comparisons, think **graph edge extraction**.

- When you have **"A must come before B"** constraints and need a valid
  total ordering, think **topological sort**.

- When a cycle in constraints makes ordering **impossible**, think
  **Kahn's BFS** — it detects cycles by comparing output length to
  node count.

- When comparing two strings for ordering clues, think **first
  differing character** — everything before it is the same in both.

- When a problem says "return any valid answer or empty string",
  think **cycle detection in directed graphs**.

## The Approach

First, collect all unique characters and build a directed graph by
scanning each consecutive word pair for the first differing character
position — that position gives an edge from word1[i] to word2[i].
Also detect the invalid prefix case (word1 longer than word2 but
word2 is a prefix of word1).

Then run Kahn's algorithm: initialize a queue with all characters
having in-degree zero, repeatedly dequeue a character into the
result, and decrement neighbors' in-degrees (adding them to the
queue when they hit zero).

Finally, if the result length matches the total number of unique
characters, return the result string; otherwise a cycle exists
and return `""`.

In [ ]:
from typing import List
from collections import defaultdict, deque

In [ ]:
def test_harness(func):
    """
    Test harness for alienOrder.
    Validates that output preserves all required ordering constraints.
    """
    def check_order(result, constraints):
        """Verify result string satisfies all (a,b) → a before b."""
        if not result:
            return len(constraints) == 0
        pos = {c: i for i, c in enumerate(result)}
        for a, b in constraints:
            if a not in pos or b not in pos:
                return False
            if pos[a] >= pos[b]:
                return False
        return True

    tests = [
        # (words, constraints, expected_empty, desc)
        (
            ["wrt", "wrf", "er", "ett", "rftt"],
            [("t","f"),("w","e"),("r","t"),("e","r")],
            False,
            "classic example"
        ),
        (
            ["z", "x"],
            [("z","x")],
            False,
            "two chars"
        ),
        (
            ["z", "x", "z"],
            [],
            True,
            "cycle → empty string"
        ),
        (
            ["abc", "ab"],
            [],
            True,
            "invalid prefix → empty string"
        ),
        (
            ["a"],
            [],
            False,
            "single word"
        ),
    ]

    passed = 0
    for words, constraints, expect_empty, desc in tests:
        result = func(words)
        if expect_empty:
            ok = (result == "")
        else:
            ok = (result != "" and check_order(result, constraints))
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(f"[{status}] {desc} | got='{result}'")

    print(f"\n{passed}/{len(tests)} tests passed.")

In [ ]:
def alienOrder(words: List[str]) -> str:
    """
    Determine alien alphabet order via topological sort.

    Args:
        words: List of words sorted in alien lexicographic order.

    Returns:
        String of characters in alien alphabetical order,
        or "" if the ordering is impossible (cycle or invalid).

    Approach:
        1. Collect all unique characters, init in_degree=0.
        2. Compare adjacent word pairs: find first diff char.
           - If word1 is longer prefix of word2 → return "".
           - Else add edge word1[i] -> word2[i] if not duplicate.
        3. Kahn's BFS: queue all in_degree=0 nodes.
        4. Process queue: append char to result, decrement
           neighbors' in_degree, enqueue if they hit 0.
        5. If len(result) == len(all chars) → return result.
           Else → cycle → return "".

    Time:  O(C) where C = total length of all words
    Space: O(1) — at most 26 chars in graph

    Debug prints show graph edges and in-degree map.
    """
    # --- Step 1: collect all chars ---
    in_degree = {c: 0 for w in words for c in w}
    graph = defaultdict(list)

    # --- Step 2: build edges from adjacent word pairs ---
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i + 1]
        min_len = min(len(w1), len(w2))
        # Invalid prefix check
        if len(w1) > len(w2) and w1[:min_len] == w2[:min_len]:
            print(f"[DEBUG] Invalid: '{w1}' prefix violation vs '{w2}'")
            return ""
        for j in range(min_len):
            if w1[j] != w2[j]:
                print(f"[DEBUG] Edge: {w1[j]} -> {w2[j]}")
                graph[w1[j]].append(w2[j])
                in_degree[w2[j]] += 1
                break  # only first differing char

    print(f"[DEBUG] in_degree: {in_degree}")

    # --- Step 3: Kahn's BFS ---
    queue = deque([c for c in in_degree if in_degree[c] == 0])
    result = []

    while queue:
        char = queue.popleft()
        result.append(char)
        for neighbor in graph[char]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    # --- Step 4: cycle check ---
    if len(result) != len(in_degree):
        print("[DEBUG] Cycle detected — returning ''")
        return ""

    return "".join(result)

In [ ]:
# Uncomment and run when solution is ready
# test_harness(alienOrder)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (try all permutations) | O(26! × C) | O(26) | Infeasible |
| Topological sort (Kahn's BFS) | O(C) | O(1)* | *26-char alphabet cap |

Where **C** = total characters across all words.

The graph has at most 26 nodes and 26×25/2 edges (bounded by
the English alphabet), so graph operations are effectively O(1).
The dominant cost is scanning all word pairs to extract edges.

## Real World Connection

At **Citi**, schema migration scripts must run in a specific order
because later migrations depend on earlier ones — exactly a
topological sort problem where a cycle means circular dependencies
that must be broken before deployment.

In **AWS Glue** and **Step Functions**, pipeline DAGs encode task
dependencies; the scheduler runs Kahn's algorithm internally to
find a valid execution order and detect deadlocks before they
waste compute resources.

For **data engineering**, dbt models form a dependency graph:
staging → intermediate → mart layers. Any circular reference
would prevent the graph from being sorted, flagged exactly like
the alien dictionary cycle return.

Understanding how to extract ordering constraints from implicit
evidence (like sorted word lists) is a core skill when reverse-
engineering undocumented data lineage in legacy systems.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra